In [8]:
#! python -m pip install numpy scipy matplotlib
import scipy as scp
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import time
from IPython.display import clear_output

In [9]:
csi_ex = np.array([
    # Paquete 1 (t = 0)
    [13.2 + 7.2j,  5.4 + 14.0j, -6.2 + 13.6j, -14.1 + 5.0j,
    -13.4 - 6.6j, -4.6 - 14.3j,  7.0 - 13.3j,  14.4 - 4.2j],
    
    # Paquete 2 (t = 1)
    [11.7 + 9.3j,  3.2 + 14.6j, -8.3 + 12.5j, -14.7 + 3.0j,
    -11.8 - 9.2j, -2.5 - 14.8j,  8.8 - 12.1j,  14.8 - 2.3j],
    
    # Paquete 3 (t = 2)
    [ 9.9 + 11.2j,  0.9 + 15.0j, -10.2 + 11.0j, -14.9 + 0.9j,
    -9.9 - 11.2j, -0.3 - 15.0j,  10.4 - 10.8j,  14.9 - 0.3j]
])

fases = []#lpm


In [ ]:
tamaño_ventana = 500
índice_renovación = 20
csi = deque(maxlen=tamaño_ventana)  # Elimina del principio cuando supera los 500
paquetes = 0

print("Iniciando visualización en tiempo real del buffer...")

while True:
    # 1. GENERAR Y AGREGAR NUEVO PAQUETE AL FINAL
    paquete_nuevo = np.random.randn(64) + 1j * np.random.randn(64)
    csi.append(paquete_nuevo)
    paquetes += 1

    time.sleep(0.01)  # Simula 100 Hz reales

    # Llenar el buffer inicial
    if len(csi) < tamaño_ventana:
        if paquetes % 20 == 0:
            print(
                f"paquete inicial... {len(csi)}/{tamaño_ventana} paquetes."
            )
        continue
    
    if paquetes % índice_renovación == 0:
        csi_largo = np.array(csi)

        print(f"paquete n°: {paquetes}")
        print("Muestra del primer paquete (csi_largo[0, :3]):")
        print(f"  -> {csi_largo[0, :3]}")
        print("Muestra del último paquete (csi_largo[-1, :3]):")
        print(f"  -> {csi_largo[-1, :3]}")

        fases_brutas = np.angle(csi_largo)
        fases_unwrapped = np.unwrap(fases_brutas, axis=0)

        fs = 100 #asumo que la frecuencia de lo que me mande hardware será 100 paquetes x seg
        nyquist = fs / 2 # tiene que ser la mitad de lo que recibe pq si no tosquea x alguna razon. se llama limite de nyquist
        low, high = 0.1 / nyquist, 0.5 / nyquist  # 0.1 son 6rpm y 0.5 30rpm
        b, a = scp.signal.butter(N=2, Wn=[low, high], btype="bandpass") #filtro butterworth, acheo todo lo quees muy alto o muy bajo
        fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0)

        varianzas = np.var(fases_filtradas, axis=0)
        mejor_subportadora = np.argmax(varianzas)
        mejor_señal = fases_filtradas[:, mejor_subportadora]

        n_fft = 10000 #resolución + fina
        fft_valores = np.fft.fft(mejor_señal, n=n_fft)
        magnitudes = np.abs(fft_valores[: n_fft // 2])
        frecuencias = np.fft.fftfreq(n_fft, d=1 / fs)[: n_fft // 2]
        rpm_pos = frecuencias * 60.0

        margen_humana = (rpm_pos >= 6.0) & (rpm_pos <= 30.0)
        rpm_validas = rpm_pos[margen_humana]
        magnitudes_validas = magnitudes[margen_humana]

        if len(magnitudes_validas) > 0:
            indice_pico = np.argmax(magnitudes_validas)
            pico_potencia = magnitudes_validas[indice_pico]
            promedio_ruido = np.mean(magnitudes)

            if pico_potencia > (3.0 * promedio_ruido):
                rpm_detectadas = rpm_validas[indice_pico]
                print(
                    f"PRESENCIA: {rpm_detectadas:.1f} RPM (Subportadora {mejor_subportadora})"
                )
            else:
                print("SIN PRESENCIA DETECTADA")
 #debería poner algo para ver que sea continuo y que se vaya mostrando el cambio constante. así se ve el rpm en cada momento y aparte si fue un ruido en el rango pero que ocurrio una vez lo saco


Iniciando visualización en tiempo real del buffer...
paquete inicial... 50/500 paquetes.
paquete inicial... 100/500 paquetes.
paquete inicial... 150/500 paquetes.
paquete inicial... 200/500 paquetes.
paquete inicial... 250/500 paquetes.
paquete inicial... 300/500 paquetes.
paquete inicial... 350/500 paquetes.
paquete inicial... 400/500 paquetes.
paquete inicial... 450/500 paquetes.
paquete n°: 500
Muestra del primer paquete (csi_largo[0, :3]):
  -> [ 1.04442183-0.8000939j  -0.95490425-1.06366416j -1.14155585+1.27096849j]
Muestra del último paquete (csi_largo[-1, :3]):
  -> [ 1.88260042+0.67020529j  0.95696337-1.47283932j -2.58144465-0.94057292j]
PRESENCIA: 12.6 RPM (Subportadora 28)
paquete n°: 520
Muestra del primer paquete (csi_largo[0, :3]):
  -> [ 0.36941587-0.04956231j -0.87313962+0.91989287j -2.19511953-0.79671272j]
Muestra del último paquete (csi_largo[-1, :3]):
  -> [-0.23819992+0.17181961j  0.49930427-0.6423702j   0.35788943-1.09810507j]
PRESENCIA: 10.8 RPM (Subportadora 33)
p

KeyboardInterrupt: 